# 06 · Cluster com 3 nós, replicação, multi-tenancy e observabilidade

O mesmo cluster usado nos notebooks anteriores (`cd docker && docker compose up -d`):

- **Weaviate**: `weaviate-0` em `:8080` / `:50051` (os outros em `:8081`, `:8082`)
- **Grafana**: http://localhost:3000, pasta *Weaviate* (dashboards de [weaviate/grafana-dashboard-weaviate#1](https://github.com/weaviate/grafana-dashboard-weaviate/pull/1))
- **Prometheus**: http://localhost:9090

In [1]:
import subprocess
import time

import pandas as pd
import requests
import weaviate
from weaviate.classes.config import Configure, DataType, Property
from weaviate.classes.query import Filter
from weaviate.classes.tenants import Tenant, TenantActivityStatus
import utils


def nos_saudaveis():
    return sum(n.status == "HEALTHY" for n in client.cluster.nodes())


client = weaviate.connect_to_local()  # fala com o weaviate-0; qualquer nó atende qualquer pedido
for n in client.cluster.nodes():
    print(n.name, n.status, n.version)

/Users/dudanogueira/dev/weaviate/lab/claude/latinoware-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


weaviate-0 HEALTHY 1.40.0
weaviate-1 HEALTHY 1.40.0
weaviate-2 HEALTHY 1.40.0


## 1. Sharding + replicação

- **Sharding** divide os dados entre os nós: mais capacidade (escala horizontal de escrita e armazenamento).
- **Replicação** copia cada shard em `factor` nós: mais disponibilidade e mais vazão de leitura.
- O **schema** é replicado via **RAFT** (consenso); os **dados** usam replicação sem líder, e a **replicação assíncrona** mantém as réplicas em dia em segundo plano.

In [2]:
client.collections.delete("WikiCluster")
wiki = client.collections.create(
    "WikiCluster",
    properties=[Property(name="title", data_type=DataType.TEXT), Property(name="text", data_type=DataType.TEXT)],
    vector_config=Configure.Vectors.self_provided(quantizer=Configure.VectorIndex.Quantizer.rq(bits=8)),
    sharding_config=Configure.sharding(desired_count=3),
    replication_config=Configure.replication(factor=3, async_enabled=True),
)

dados = utils.load_wiki(1)
inicio = time.perf_counter()
with wiki.batch.fixed_size(batch_size=500) as batch:
    for linha in dados.itertuples():
        batch.add_object({"title": linha.title, "text": linha.text}, vector=linha.vector)
print(f"{len(dados):,} objetos em {time.perf_counter() - inicio:.1f}s | falhas: {len(wiki.batch.failed_objects)}")
utils.wait_for_indexing(client, "WikiCluster")

25,000 objetos em 5.6s | falhas: 0


In [3]:
linhas = [
    {"nó": n.name, "shard": s.name, "réplica": "✅"}
    for n in client.cluster.nodes(collection="WikiCluster", output="verbose")
    for s in n.shards
]
print("objetos na collection:", len(wiki))
pd.DataFrame(linhas).pivot(index="shard", columns="nó", values="réplica")

objetos na collection: 25000


nó,weaviate-0,weaviate-1,weaviate-2
shard,,,
R1zRxtCRVd3q,✅,✅,✅
VHiKEdOuQoUI,✅,✅,✅
vwd6ML4x2cev,✅,✅,✅


3 shards × 3 réplicas: cada nó guarda uma cópia de cada shard. Uma busca vai para todos os shards e junta os resultados.

## 2. Derrubando um nó ao vivo 💥

In [4]:
pergunta = utils.embed_query("como os vulcões entram em erupção?")


def buscar():
    r = wiki.query.near_vector(pergunta, limit=1)
    return r.objects[0].properties["title"]


print("3 nós no ar ->", buscar())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7762.71it/s]

3 nós no ar -> Volcanism


In [5]:
subprocess.run(["docker", "stop", "latinoware-cluster-weaviate-2-1"], check=True, capture_output=True)
while nos_saudaveis() == 3:  # o nó parado some da lista
    time.sleep(1)
print("nós:", [(n.name, n.status) for n in client.cluster.nodes()])
print("1 nó fora ->", buscar())

nós: [('weaviate-0', 'HEALTHY'), ('weaviate-1', 'HEALTHY')]
1 nó fora -> Volcanism


O cluster continua respondendo com um nó a menos. E continua aceitando escritas:

In [6]:
novo = wiki.data.insert(
    {"title": "Latinoware 2026", "text": "Inserido enquanto o weaviate-2 estava fora do ar."},
    vector=utils.embed_query("Latinoware 2026, Foz do Iguaçu"),
)
print("inserido:", novo)

inserido: 3a52ce3e-7388-4557-a2fd-912d1d967294


Quando o nó volta, a **replicação assíncrona** compara as réplicas (árvores de hash) e leva ao `weaviate-2` o que ele perdeu, sem intervenção manual. Leva algumas dezenas de segundos: as réplicas trocam *hash trees* e só transferem o que difere.

In [7]:
subprocess.run(["docker", "start", "latinoware-cluster-weaviate-2-1"], check=True, capture_output=True)
while nos_saudaveis() < 3:
    time.sleep(1)
print("3 nós de volta ->", buscar())

3 nós de volta -> Volcanism


In [8]:
def no2_tem_o_objeto():
    try:
        with weaviate.connect_to_local(port=8082, grpc_port=50053) as no2:  # direto no weaviate-2
            return no2.collections.use("WikiCluster").query.fetch_object_by_id(novo)
    except weaviate.exceptions.WeaviateBaseError:
        return None  # nó ainda terminando de subir


inicio = time.perf_counter()
while (obj := no2_tem_o_objeto()) is None:
    time.sleep(2)
print(f"weaviate-2 recebeu '{obj.properties['title']}' {time.perf_counter() - inicio:.0f}s depois de voltar")

# ## 3. Multi-tenancy: um shard por cliente
#
# Em SaaS, cada cliente (tenant) ganha seu próprio shard: isolamento de dados, deleção trivial (LGPD!) e controle de recursos. Tenants **inativos** saem da memória; com o módulo `offload-s3`, vão para o S3 (**frozen**).

weaviate-2 recebeu 'Latinoware 2026' 29s depois de voltar


In [9]:
client.collections.delete("Notas")
notas = client.collections.create(
    "Notas",
    properties=[Property(name="texto", data_type=DataType.TEXT)],
    vector_config=Configure.Vectors.self_provided(vector_index_config=Configure.VectorIndex.dynamic()),
    multi_tenancy_config=Configure.multi_tenancy(enabled=True, auto_tenant_creation=True, auto_tenant_activation=True),
    replication_config=Configure.replication(factor=2),
)
notas.tenants.create([Tenant(name=f"cliente-{i:03d}") for i in range(100)])

amostra = dados.sample(500, random_state=0)
for i, linha in enumerate(amostra.itertuples()):
    notas.with_tenant(f"cliente-{i % 100:03d}").data.insert({"texto": linha.text}, vector=linha.vector)

cliente7 = notas.with_tenant("cliente-007")
print("objetos do cliente-007:", len(cliente7))
print("busca só no cliente-007:", cliente7.query.near_vector(pergunta, limit=1).objects[0].properties["texto"][:80], "…")

objetos do cliente-007: 5
busca só no cliente-007: Each year many people attempt (try) to do the challenge to raise money for chari …


In [10]:
notas.tenants.update([Tenant(name=f"cliente-{i:03d}", activity_status=TenantActivityStatus.INACTIVE) for i in range(50, 100)])
status = pd.Series({t.name: t.activity_status.value for t in notas.tenants.get().values()}).value_counts()
status

ACTIVE      50
INACTIVE    50
Name: count, dtype: int64

In [11]:
print("tenants ativos carregados por nó (replication factor 2):")
for n in client.cluster.nodes(collection="Notas", output="verbose"):
    print(f"  {n.name}: {len(n.shards)} shards")

tenants ativos carregados por nó (replication factor 2):
  weaviate-0: 33 shards
  weaviate-1: 31 shards
  weaviate-2: 36 shards


Com `auto_tenant_activation`, um tenant inativo volta sozinho no primeiro acesso.

## 4. Observabilidade

Cada nó expõe métricas Prometheus em `:2112/metrics` (`PROMETHEUS_MONITORING_ENABLED=true`). Primeiro, um pouco de carga em segundo plano:

In [12]:
import threading

parar = threading.Event()


def carga():
    with weaviate.connect_to_local() as c:
        w = c.collections.use("WikiCluster")
        vetores = dados.vector.sample(200, random_state=1).tolist()
        while not parar.is_set():
            for v in vetores:
                w.query.near_vector(v, limit=10)
                w.query.bm25("volcano", limit=10)
                if parar.is_set():
                    break


threading.Thread(target=carga, daemon=True).start()
time.sleep(45)

As métricas que mais importam no dia a dia:

In [13]:
PROM = "http://localhost:9090/api/v1/query"
consultas = {
    "nós no ar": 'count(up{job="weaviate"} == 1)',
    "objetos (todas as réplicas)": "sum(object_count)",
    "consultas/s": 'sum(rate(queries_durations_ms_count[1m]))',
    "latência p95 por tipo (ms)": 'histogram_quantile(0.95, sum by (le, query_type) (rate(queries_durations_ms_bucket[1m])))',
    "fila de indexação assíncrona": "sum(queue_size)",
    "heap Go por nó (MB)": "go_memstats_heap_inuse_bytes / 1e6",
    "goroutines por nó": "go_goroutines",
    "falhas de compactação LSM": "sum(weaviate_lsm_bucket_compaction_failure_count) or vector(0)",
}
for nome, q in consultas.items():
    for r in requests.get(PROM, params={"query": q}).json()["data"]["result"]:
        rotulo = r["metric"].get("query_type") or r["metric"].get("instance", "")
        print(f"{nome:30} {rotulo:20} {float(r['value'][1]):,.1f}")

nós no ar                                           3.0
objetos (todas as réplicas)                         918,700.0
consultas/s                                         455.1
latência p95 por tipo (ms)     get_graphql          9.5
fila de indexação assíncrona                        18.0
heap Go por nó (MB)            weaviate-2:2112      1,329.1
heap Go por nó (MB)            weaviate-1:2112      1,511.7
heap Go por nó (MB)            weaviate-0:2112      1,383.9
goroutines por nó              weaviate-2:2112      200.0
goroutines por nó              weaviate-1:2112      200.0
goroutines por nó              weaviate-0:2112      217.0
falhas de compactação LSM                           0.0


No Grafana, os mesmos dados com histórico:

- **Overview**: http://localhost:3000/d/weaviate-overview
- **Querying**: http://localhost:3000/d/ImxRLXe7z
- **Async Indexing**: http://localhost:3000/d/cdnvmrjxw1ds0d
- **Cluster & RAFT**: http://localhost:3000/d/weaviate-cluster
- **Replication**: http://localhost:3000/d/aejvwfv8vkq2oa

**O que alertar em produção:** nó fora (`up == 0`), latência p95/p99 acima do SLO, fila de indexação crescendo sem parar, heap perto do `GOMEMLIMIT`, falhas de compactação/flush do LSM, disco acima de ~80%.

In [14]:
parar.set()
client.close()